In [5]:
import requests
import pandas as pd
from datetime import datetime, timezone
import time
import random
from typing import Any
import json
from pathlib import Path

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# Funciones obtención .json de API BOE

- Cada día guardo la respuesta completa obtenida de la API del BOE para esa fecha. Guardar en S3 bronze.
- Controlar días sin BOE o errores 404.
- Concatenar los DataFrame diarios y guardar el resultado en CSV/Parquet.

In [6]:
def as_list(value):
    """
    Convierte un valor en una lista.

    Esta función se utiliza para normalizar la estructura de los datos
    devueltos por la API del BOE, donde un mismo nodo puede aparecer
    como un único objeto (dict) o como una lista de objetos (list).

    Parámetros
    ----------
    value : any
        Valor a normalizar.

    Retorna
    -------
    list
        - [] si value es None.
        - value si ya es una lista.
        - [value] si es un único elemento.
    """
    if value is None:
        return []

    return value if isinstance(value, list) else [value]

In [7]:
def get_sumario_boe(fecha: str) -> dict[str, Any]:
    """
    Obtiene el sumario del BOE correspondiente a una fecha determinada.

    Realiza una petición HTTP GET a la API de datos abiertos del BOE y
    devuelve la respuesta en formato JSON ya deserializada como un
    diccionario de Python.

    La fecha debe especificarse en formato AAAAMMDD. Por ejemplo:
    - 20230102 → BOE de 2 de enero de 2023
    - 20240529 → BOE de 29 de mayo de 2024

    Parámetros
    ----------
    fecha : str
        Fecha de publicación del BOE en formato AAAAMMDD.

    Retorna
    -------
    dict
        Respuesta JSON devuelta por la API del BOE.

    Raises
    ------
    requests.exceptions.HTTPError
        Si la API devuelve un código de error HTTP
        (por ejemplo, 404 si no existe el sumario solicitado).

    requests.exceptions.RequestException
        Si ocurre cualquier problema de comunicación con el servidor
        (timeout, error de conexión, etc.).

    Ejemplos
    --------
    >>> data = get_sumario_boe("20230102")
    >>> data["status"]["code"]
    "200"
    """
    url = f"{BASE_URL}/{fecha}"

    response = requests.get(
        url,
        headers={"Accept": "application/json"},
        timeout=30,
    )

    response.raise_for_status()

    return response.json()

In [8]:
def save_json_local(data: dict[str, Any], path: Path) -> None:
    """
    Guarda un diccionario Python como archivo JSON local.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

In [9]:
def count_items_sumario_boe(data: dict[str, Any]) -> int:
    """
    Cuenta de forma aproximada los ítems publicados en un sumario BOE.
    """
    count = 0

    sumario = data.get("data", {}).get("sumario", {})
    diarios = as_list(sumario.get("diario"))

    for diario in diarios:
        for seccion in as_list(diario.get("seccion")):
            for departamento in as_list(seccion.get("departamento")):

                for epigrafe in as_list(departamento.get("epigrafe")):
                    count += len(as_list(epigrafe.get("item")))

                texto = departamento.get("texto")
                if isinstance(texto, dict):
                    count += len(as_list(texto.get("item")))

                count += len(as_list(departamento.get("item")))

    return count

In [14]:
def ingest_sumario_boe_local(
    fecha: str,
    base_dir: Path = Path(BRONZE_DIR),
) -> dict[str, Any]:
    """
    Descarga el sumario diario del BOE y lo guarda en local.
    """
    day_dir = base_dir / fecha

    metadata: dict[str, Any] = {
        "date": fecha,
        "source": "BOE API sumario",
        "ingested_at": datetime.now(timezone.utc).isoformat(),
        "status": None,
        "http_status": None,
        "boe_status_code": None,
        "records_downloaded": 0,
        "error_type": None,
        "error": None,
    }

    try:
        data = get_sumario_boe(fecha)

        boe_status_code = data.get("status", {}).get("code")
        metadata["boe_status_code"] = boe_status_code
        metadata["http_status"] = 200

        if boe_status_code != "200":
            metadata["status"] = "failed"
            metadata["error_type"] = "BOE_STATUS_ERROR"
            metadata["error"] = data.get("status", {}).get("text")
            save_json_local(metadata, day_dir / "metadata.json")
            return metadata

        save_json_local(data, day_dir / "sumario.json")

        metadata["status"] = "success"
        metadata["records_downloaded"] = count_items_sumario_boe(data)

        save_json_local(metadata, day_dir / "metadata.json")
        return metadata

    except requests.exceptions.HTTPError as exc:
        response = exc.response
        http_status = response.status_code if response is not None else None

        metadata["http_status"] = http_status

        if http_status == 404:
            metadata["status"] = "no_publication"
            metadata["error_type"] = "HTTP_404"
            metadata["error"] = "No BOE publication for this date"
        else:
            metadata["status"] = "failed"
            metadata["error_type"] = "HTTP_ERROR"
            metadata["error"] = str(exc)

        save_json_local(metadata, day_dir / "metadata.json")
        return metadata

    except requests.exceptions.RequestException as exc:
        metadata["status"] = "failed"
        metadata["error_type"] = "REQUEST_ERROR"
        metadata["error"] = str(exc)
        save_json_local(metadata, day_dir / "metadata.json")
        return metadata

    except ValueError as exc:
        metadata["status"] = "failed"
        metadata["error_type"] = "INVALID_JSON"
        metadata["error"] = str(exc)
        save_json_local(metadata, day_dir / "metadata.json")
        return metadata

In [15]:
fechas = ["20230102"]

for fecha in fechas:
    data = get_sumario_boe(fecha)

    time.sleep(random.uniform(0.2, 1.5))
    
data

{'status': {'code': '200', 'text': 'ok'},
 'data': {'sumario': {'metadatos': {'publicacion': 'BOE',
    'fecha_publicacion': '20230102'},
   'diario': [{'numero': '1',
     'sumario_diario': {'identificador': 'BOE-S-2023-1',
      'url_pdf': {'szBytes': '371451',
       'szKBytes': '363',
       'texto': 'https://www.boe.es/boe/dias/2023/01/02/pdfs/BOE-S-2023-1.pdf'}},
     'seccion': [{'codigo': '2A',
       'nombre': 'II. Autoridades y personal. - A. Nombramientos, situaciones e incidencias',
       'departamento': [{'codigo': '9562',
         'nombre': 'MINISTERIO DE ASUNTOS EXTERIORES, UNIÓN EUROPEA Y COOPERACIÓN',
         'epigrafe': [{'nombre': 'Destinos',
           'item': {'identificador': 'BOE-A-2023-1',
            'control': '2022/23843',
            'titulo': 'Resolución de 23 de diciembre de 2022, de la Subsecretaría, por la que se resuelve parcialmente la convocatoria de libre designación, efectuada por Resolución de 27 de octubre de 2022.',
            'url_pdf': {'szB

In [16]:
metadata = ingest_sumario_boe_local("20230102")
print(metadata)

{'date': '20230102', 'source': 'BOE API sumario', 'ingested_at': '2026-06-11T12:27:47.873248+00:00', 'status': 'success', 'http_status': 200, 'boe_status_code': '200', 'records_downloaded': 164, 'error_type': None, 'error': None}


# Funciones de parseo

In [ ]:
def parse_item(item: dict[str, Any], seccion: str, departamento: str, epigrafe: str | None, dt: datetime) -> dict[str, Any]:
    """
    Transforma una disposición o anuncio del BOE en un registro plano.

    Convierte la estructura jerárquica devuelta por la API de Sumarios
    del BOE en un diccionario con los campos normalizados que formarán
    parte del conjunto de datos final.

    Además de extraer los metadatos de la disposición, incorpora
    información contextual procedente de la sección, el departamento
    y la fecha de publicación.

    Parámetros
    ----------
    item : dict
        Nodo <item> de la API del BOE. Contiene la información de una
        disposición o anuncio publicado en el diario.

    seccion : str
        Nombre de la sección del BOE a la que pertenece la disposición.
        Ejemplo: "I. Disposiciones generales".

    departamento : str
        Nombre del departamento u organismo responsable de la publicación.
        Ejemplo: "MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL RETO
        DEMOGRÁFICO".

    epigrafe : str | None
        Nombre del epígrafe al que pertenece el item. Será None cuando
        el item esté directamente dentro del nodo departamento.

    dt : datetime
        Fecha de publicación del BOE.

    Retorna
    -------
    dict
        Diccionario con la estructura normalizada del dataset:

        - identificador
        - titulo
        - url_html
        - pdf_link
        - seccion
        - departamento
        - epigrafe
        - site
        - place
        - date
        - year
        - month
        - day

    Notes
    -----
    El campo 'url_pdf' puede aparecer como una cadena o como un
    diccionario con metadatos adicionales (tamaño, páginas, etc.).
    En ambos casos se extrae únicamente la URL del documento PDF.

    Los campos 'site' y 'place' son valores constantes añadidos para
    facilitar la integración con otros conjuntos de datos.
    """
    url_pdf = item.get("url_pdf")

    return {
        "identificador": item.get("identificador"),
        "titulo": item.get("titulo"),
        "url_html": item.get("url_html"),
        "pdf_link": url_pdf.get("texto") if isinstance(url_pdf, dict) else url_pdf,
        "seccion": seccion,
        "departamento": departamento,
        "epigrafe": epigrafe,
        "site": "boe",
        "place": "espana",
        "date": dt.strftime("%Y/%m/%d"),
        "year": dt.year,
        "month": dt.month,
        "day": dt.day,
    }

In [ ]:
def parse_sumario_boe(data: dict[str, Any]) -> pd.DataFrame:
    """
    Convierte la respuesta JSON de la API de Sumarios del BOE en un DataFrame.

    Recorre la estructura jerárquica del sumario y extrae cada disposición
    o anuncio publicado junto con su contexto: sección, departamento y,
    cuando exista, epígrafe.

    La función contempla items ubicados dentro de ``epigrafe`` e items
    ubicados directamente dentro de ``departamento``.

    Parámetros
    ----------
    data : dict[str, Any]
        Respuesta JSON devuelta por la API de Sumarios del BOE.

    Retorna
    -------
    pd.DataFrame
        DataFrame con una fila por disposición o anuncio publicado.
    """
    rows: list[dict[str, Any]] = []

    try:
        sumario = data["data"]["sumario"]
    except KeyError as exc:
        raise ValueError(
            "La respuesta no contiene la estructura esperada: data.sumario."
        ) from exc

    fecha = sumario.get("metadatos", {}).get("fecha_publicacion")

    if not fecha:
        raise ValueError(
            "La respuesta no contiene 'fecha_publicacion' en los metadatos."
        )

    try:
        dt = datetime.strptime(fecha, "%Y%m%d")
    except ValueError as exc:
        raise ValueError(
            f"La fecha_publicacion no tiene formato AAAAMMDD válido: {fecha}"
        ) from exc

    for diario in as_list(sumario.get("diario")):
        for seccion in as_list(diario.get("seccion")):
            seccion_nombre = seccion.get("nombre")

            for departamento in as_list(seccion.get("departamento")):
                departamento_nombre = departamento.get("nombre")

                for epigrafe in as_list(departamento.get("epigrafe")):
                    epigrafe_nombre = epigrafe.get("nombre")

                    for item in as_list(epigrafe.get("item")):
                        rows.append(
                            parse_item(
                                item=item,
                                seccion=seccion_nombre,
                                departamento=departamento_nombre,
                                epigrafe=epigrafe_nombre,
                                dt=dt,
                            )
                        )

                for item in as_list(departamento.get("item")):
                    rows.append(
                        parse_item(
                            item=item,
                            seccion=seccion_nombre,
                            departamento=departamento_nombre,
                            epigrafe=None,
                            dt=dt,
                        )
                    )

    return pd.DataFrame(rows)

In [ ]:
df = parse_sumario_boe(data)
display(df.head(10))
display(df.loc[df["identificador"] == "BOE-A-2023-49"])
